In [0]:
%run "../libs/libs"

In [0]:
from pyspark.sql.functions import current_timestamp, lit, col, to_date

In [0]:
order_df = spark.read.format("delta").table("dev.tracking_sl.order")
dimdate_df = spark.read.format("delta").table("dev.tracking_gd.dimdate")
org_df = spark.read.format("delta").table("dev.tracking_gd.dimorganization")
vehicle_df = spark.read.format("delta").table("dev.tracking_gd.dimvehicle")


In [0]:
sender = org_df.alias('sender')
receiver = org_df.alias('receiver')
order = order_df.alias('order')


tdetail_df = (
    order.join(
        dimdate_df,
        to_date(order.orderdate) == dimdate_df['date'],
        "inner"
    )
    .join(
        vehicle_df,
        (order.carriername == vehicle_df['carrriername']) &
        (order.carriercity == vehicle_df['carriercity']) &
        (order.carrierstate == vehicle_df['carrierstate']),
        "inner"
    )
    .join(
        sender,
        (order.sendername == sender['name']) &
        (order.sendercity == sender['city']) &
        (order.senderstate == sender['state']),
        "inner"
    )   
    .join(
        receiver,
        (order.receivername == receiver['name']) &
        (order.receivercity == receiver['city']) &
        (order.receiverstate == receiver['state']),
        "inner"
    )      
.select(
        col('orderno'),
        col('datekey'),
        col('shipmentno'),
        col('vehicleid').alias('vehicle_id'),
        col('sender.orgid').alias('sender_id'),
        col('receiver.orgid').alias('receiver_id')
    )
    .withColumn('ingestion_time', current_timestamp())
)


In [0]:
tdetail_df.createOrReplaceTempView("temp_dimorder")

In [0]:
%sql
merge into dev.tracking_gd.dimorder tgt
USING ( 
  SELECT DISTINCT * 
  FROM temp_dimorder 
) src 
on tgt.orderno = src.orderno
when not matched then
  insert (orderno, datekey, shipmentno,vehicle_id,sender_id,receiver_id, ingestion_time)
  Values(src.orderno,src.datekey,src.shipmentno,src.vehicle_id,src.sender_id,src.receiver_id,src.ingestion_time)

In [0]:
%sql
select * from dev.tracking_gd.dimorder limit 100